Importation des données 

In [5]:
from sklearn.datasets import fetch_openml

freq = fetch_openml(data_id=41214, as_frame=True).frame
sev = fetch_openml(data_id=41215, as_frame=True).frame

print(freq.head())
print(sev.head())
print(freq.shape)
print(sev.shape)

   IDpol  ClaimNb  Exposure Area  VehPower  VehAge  DrivAge  BonusMalus  \
0    1.0        1      0.10    D         5       0       55          50   
1    3.0        1      0.77    D         5       0       55          50   
2    5.0        1      0.75    B         6       2       52          50   
3   10.0        1      0.09    B         7       0       46          50   
4   11.0        1      0.84    B         7       0       46          50   

  VehBrand     VehGas  Density Region  
0      B12  'Regular'     1217    R82  
1      B12  'Regular'     1217    R82  
2      B12   'Diesel'       54    R22  
3      B12   'Diesel'       76    R72  
4      B12   'Diesel'       76    R72  
     IDpol  ClaimAmount
0     1552       995.20
1  1010996      1128.12
2  4024277      1851.11
3  4007252      1204.00
4  4046424      1204.00
(678013, 12)
(26639, 2)


In [13]:
print((freq['ClaimNb']).value_counts().sort_index())
print((freq['ClaimNb'] > 0).mean())

ClaimNb
0     643953
1      32178
2       1784
3         82
4          7
5          2
6          1
8          1
9          1
11         3
16         1
Name: count, dtype: int64
0.05023502499214617


In [18]:
print(freq['Exposure'].describe())
print((freq['Exposure'] > 1).sum())

count    678013.000000
mean          0.528750
std           0.364442
min           0.002732
25%           0.180000
50%           0.490000
75%           0.990000
max           2.010000
Name: Exposure, dtype: float64
1224


In [19]:
print(freq['IDpol'].duplicated().sum())
print(sev['ClaimAmount'].describe())
print(sev['IDpol'].duplicated().sum())

0
count    2.663900e+04
mean     2.278536e+03
std      2.929748e+04
min      1.000000e+00
25%      6.868100e+02
50%      1.172000e+03
75%      1.228080e+03
max      4.075401e+06
Name: ClaimAmount, dtype: float64
1689


un montant de 1 euro a ete payé, ce qui parait impossible pour un sinistre automobile, il s'agit surement la d'une erreur de saisie, un artefact dans le jeu de donnée qu'il sera important de retirer avant l'entrainement des modèles.

In [23]:
print((sev["ClaimAmount"] <= 1).sum())
print(sev["ClaimAmount"].quantile([0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))
print(sev['ClaimAmount'].value_counts().head(10))

1
0.01       40.0314
0.10      197.3360
0.25      686.8100
0.50     1172.0000
0.75     1228.0800
0.90     2799.0720
0.99    16793.7044
Name: ClaimAmount, dtype: float64
ClaimAmount
1204.00    4792
1128.12    3056
1172.00    2072
1128.00     831
602.00      433
1320.00     175
556.14      155
586.00      104
1500.00     103
564.06       94
Name: count, dtype: int64


on remarque que certaines valeurs reveinnent beaucoup dans les couts d'un sinistre. Surement un montant de provision standart demandé par l'assureur à l'ouverture d'un dossier de sinistre.

Il est très important de le noter car cela aura une incidence sur la suite car ces montants etants vraissemblablement standarts et automatique, il ne dependent pas a premiere vue des caractéristiques de l'assurés, c'est un forfait.

In [20]:
print(freq['ClaimNb'].sum() / freq['Exposure'].sum())

0.10070308464041305


la frequence du nombre de sinistre est de 0.1 par an, ce qui me parait beaucouop

In [25]:
print(freq['Exposure'].max())
print((freq['Exposure'] > 1).sum())
print(freq['IDpol'].duplicated().sum())

2.01
1224
0


Tache 1

la table freq contient les données de contrats (678013) et la table sev celles de sinistres (26639) donc beaucoup plus de sinistres que de contrats (logique)

les deux tables sont relié par un identifiant unique a chaque client : IDpol. Il peut donc y avoir plusieurs sinistres par contrats.

On a une feature Exposure dans la table freq qui correspond à l'exposition en année police.

34060 on eu au moins 1 sinistres ce qui représente 5%, ce qui me parait normal. apres il faut voir sur quelle moyenne de temps de contrat. Sur un moyenne assez longue oui mais sur un courte cela devient suspect.

Le minimum d'exposition est 0.002732 ce qui représente seulement quelque jours, si il y a eu un sinistre cela pose vraiment question

il y a 1689 assuré qui ont eu plus de 1 sinistres

le plus gros sinistre (en couts) est de 4 000 000

